In [ ]:
import sqlite3
import warnings
import logging
from pprint import pprint
from typing import Any, Dict

from google.adk.agents import Agent, LlmAgent
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.models.google_llm import Gemini
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

from google.adk.events import Event
from google.adk.sessions import Session
from typing import Optional

class NoFunctionCallWarningFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        return "there are non-text parts in the response" not in record.getMessage()
        
logger = logging.getLogger("google_genai.types")
logger.addFilter(NoFunctionCallWarningFilter())

warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
class FixedDatabaseSessionService(DatabaseSessionService):
    """
    A more robust patch for the Database/Compaction bug.
    
    This version manually re-hydrates the main Event object AND
    the nested Event.actions.compaction object, which is the
    source of the 'start_timestamp' error.
    """
    async def get_session(
        self, session_id: str, app_name: str, user_id: Optional[str] = None
    ) -> Optional[Session]:
        
        # 1. Get the session (which has session.events as list[dict])
        session = await super().get_session(
            session_id=session_id, 
            app_name=app_name, 
            user_id=user_id
        )

        if not session or not session.events:
            return session

        # 2. This is the new, deeper fix.
        try:
            rehydrated_events = []
            for evt_dict in session.events:
                if not isinstance(evt_dict, dict):
                    if isinstance(evt_dict, Event):
                        rehydrated_events.append(evt_dict) # Already an Event
                    continue # Skip junk
                
                # --- MANUAL NESTED FIX ---
                # This is the key: we find the nested compaction dict
                # and re-hydrate it *before* validating the parent Event.
                if (
                    "actions" in evt_dict and
                    evt_dict["actions"] and 
                    "compaction" in evt_dict["actions"] and 
                    isinstance(evt_dict["actions"]["compaction"], dict)
                ):
                    try:
                        # Convert the compaction 'dict' into a 'CompactionAction' object
                        compaction_dict = evt_dict["actions"]["compaction"]
                        evt_dict["actions"]["compaction"] = CompactionAction.model_validate(compaction_dict)
                    except Exception as e:
                        logger.warning(f"Failed to hydrate nested compaction dict: {e}")
                # --- END MANUAL FIX ---

                # Now validate the whole Event dict
                rehydrated_events.append(Event.model_validate(evt_dict))
            
            session.events = rehydrated_events
            
        except Exception as e:
            logger.error(f"CRITICAL: Failed to re-hydrate events: {e}")
            raise
        
        # 3. Return the fixed session
        return session

# Sessions

### Helper Functions

In [3]:
# function that manages a complete conversation session
# - handling session creation/retrieval
# - query processing
# - response streaming
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
):
    print(f"\n ### Session: {session_name}")

    # Get app name from the Runner
    app_name = runner_instance.app_name

    # Attempt to create a new session or retrieve an existing one
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )

    # Process queries if provided
    if user_queries:
        # Convert single query to list for uniform processing
        if type(user_queries) == str:
            user_queries = [user_queries]

        # Process each query in the list sequentially
        for query in user_queries:
            print(f"\nUser > {query}")

            # Convert the query string to the ADK Content format
            query = types.Content(role="user", parts=[types.Part(text=query)])

            # Stream the agent's response asynchronously
            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query
            ):
                # Check if the event contains valid content
                if event.content and event.content.parts:
                    # Filter out empty or "None" responses before printing
                    if (
                        event.content.parts[0].text != "None"
                        and event.content.parts[0].text
                    ):
                        print(f"{MODEL_NAME} > ", event.content.parts[0].text)
    else:
        print("No queries!")

In [4]:
# retry options
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

### Session Management

In [5]:
# Sessions for short term memory management
# Memory for for long term memory

# A session is a container for conversations
# - encapsulates the conversation history in a chronological manner
# - records all tool interactions and responses for a single, continuous conversation

# Events are the building blocks of a conversation
# - User input
# - Agent response
# - Tool call
# - Tool output

# session.state is the Agent's scratchpad, where it stores and updates dynamic details needed during the conversation

### Stateful Agent

In [6]:
APP_NAME = "agents"
USER_ID = "default"
SESSION = "default"

MODEL_NAME = "gemini-2.5-flash-lite"

# Step 1: Create the LLM Agent
root_agent = Agent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot",  # Description of the agent's purpose
)

# Step 2: Set up Session Management
# InMemorySessionService stores conversations in RAM (temporary)
session_service = InMemorySessionService()

# Step 3: Create the Runner
runner = Runner(agent=root_agent, 
                app_name=APP_NAME, 
                session_service=session_service)

In [7]:
await run_session(
    runner,
    [
        "Hi, I am Alex! What is the capital of Czech Republic?",
        "Hello! What is my name?",  # This time, the agent should remember!
    ],
    "stateful-agentic-session",
)


 ### Session: stateful-agentic-session

User > Hi, I am Alex! What is the capital of Czech Republic?
gemini-2.5-flash-lite >  Hi Alex! The capital of the Czech Republic is Prague.

User > Hello! What is my name?
gemini-2.5-flash-lite >  Your name is Alex.


In [8]:
# Restart kernel and run this instead of previous run - session is not persistent and reinitialized
await run_session(
    runner,
    ["What did I ask you about earlier?", "And remind me, what's my name?"],
    "stateful-agentic-session",
)


 ### Session: stateful-agentic-session

User > What did I ask you about earlier?
gemini-2.5-flash-lite >  You asked me about the capital of the Czech Republic.

User > And remind me, what's my name?
gemini-2.5-flash-lite >  Your name is Alex.


### Persistent Sessions

In [9]:
# upgrade to DatabaseSessionService using SQLite

# Step 1: Create the same agent (notice we use LlmAgent this time)
chatbot_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot with persistent memory",
)

# Step 2: Switch to DatabaseSessionService
# SQLite database will be created automatically
db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

# Step 3: Create a new runner with persistent storage
runner = Runner(agent=chatbot_agent, 
                app_name=APP_NAME, 
                session_service=session_service)

In [10]:
await run_session(
    runner,
    ["Hi, I am Alex! What is the capital of Czech Republic?", 
     "Hello! What is my name?"],
    "test-db-session-01",
)


 ### Session: test-db-session-01

User > Hi, I am Alex! What is the capital of Czech Republic?
gemini-2.5-flash-lite >  Hi Alex! The capital of the Czech Republic is Prague.

User > Hello! What is my name?
gemini-2.5-flash-lite >  Hello Alex! Your name is Alex.


In [11]:
# restart kernel
await run_session(
    runner,
    ["What is the capital of India?", 
     "Hello! What is my name?"],
    "test-db-session-01",
)


 ### Session: test-db-session-01

User > What is the capital of India?
gemini-2.5-flash-lite >  The capital of India is New Delhi.

User > Hello! What is my name?
gemini-2.5-flash-lite >  Your name is Alex.


In [12]:
def check_data_in_db():
    with sqlite3.connect("my_agent_data.db") as connection:
        cursor = connection.cursor()
        result = cursor.execute(
            "select app_name, session_id, author, content from events"
        )
        print([_[0] for _ in result.description])
        for each in result.fetchall():
            pprint(each)

In [13]:
check_data_in_db()

['app_name', 'session_id', 'author', 'content']
('agents',
 'test-db-session-01',
 'user',
 '{"parts": [{"text": "Hi, I am Alex! What is the capital of Czech '
 'Republic?"}], "role": "user"}')
('agents',
 'test-db-session-01',
 'text_chat_bot',
 '{"parts": [{"text": "Hi Alex! The capital of the Czech Republic is '
 'Prague."}], "role": "model"}')
('agents',
 'test-db-session-01',
 'user',
 '{"parts": [{"text": "Hello! What is my name?"}], "role": "user"}')
('agents',
 'test-db-session-01',
 'text_chat_bot',
 '{"parts": [{"text": "Hello Alex! Your name is Alex."}], "role": "model"}')
('agents',
 'test-db-session-01',
 'user',
 '{"parts": [{"text": "What is the capital of India?"}], "role": "user"}')
('agents',
 'test-db-session-01',
 'text_chat_bot',
 '{"parts": [{"text": "The capital of India is New Delhi."}], "role": "model"}')
('agents',
 'test-db-session-01',
 'user',
 '{"parts": [{"text": "Hello! What is my name?"}], "role": "user"}')
('agents',
 'test-db-session-01',
 'text_chat_

### Context Compaction

In [14]:
# For a long, complex task, this list of events can become very large, leading to slower performance and higher costs
# ADK's Context Compaction automatically reduces the context that's stored in the Session

In [15]:
# Re-define our app with Events Compaction enabled
research_app_compacting = App(
    name="agents",
    root_agent=chatbot_agent,
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=3,  # Trigger compaction every 3 invocations
        overlap_size=1,  # Keep 1 previous turn for context
    ),
)

db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = FixedDatabaseSessionService(db_url=db_url)

# Create a new runner for our upgraded app
research_runner_compacting = Runner(
    app=research_app_compacting, 
    session_service=session_service
)

In [16]:
# Turn 1
await run_session(
    research_runner_compacting,
    "What is the latest news about AI in healthcare?",
    "compaction_demo",
)


 ### Session: compaction_demo

User > What is the latest news about AI in healthcare?
gemini-2.5-flash-lite >  Here's a glimpse into some of the latest news and trends in AI in healthcare:

**1. Generative AI's Expanding Role:**

*   **Drug Discovery and Development:** Generative AI is making significant strides in identifying novel drug candidates, predicting molecular interactions, and optimizing clinical trial design. Companies are leveraging these tools to accelerate the traditionally slow and expensive process of bringing new medicines to market.
*   **Clinical Documentation and Summarization:** AI models are becoming increasingly adept at assisting clinicians with tasks like generating patient summaries, drafting clinical notes, and even creating discharge instructions. This can free up valuable physician time and reduce burnout.
*   **Personalized Treatment Plans:** Generative AI can analyze vast amounts of patient data (genomics, medical history, lifestyle) to suggest highly p

In [17]:
# Turn 2
await run_session(
    research_runner_compacting,
    "Are there any new developments in drug discovery?",
    "compaction_demo",
)


 ### Session: compaction_demo

User > Are there any new developments in drug discovery?
gemini-2.5-flash-lite >  Yes, there are very exciting and rapid developments in drug discovery, largely powered by advancements in AI and other cutting-edge technologies. Here's a breakdown of some of the newest trends and developments:

**1. AI-Powered Drug Discovery - The Dominant Force:**

*   **Generative AI for Novel Molecule Design:** This is a huge area. Instead of just screening existing compounds, generative AI models are now designing entirely *new* molecules from scratch that have desired properties. They can optimize for factors like efficacy, safety, and manufacturability.
    *   **Example:** Companies are using AI to design novel antibiotics, cancer therapies, and treatments for rare diseases.
*   **Accelerated Target Identification:** AI can sift through massive datasets (genomic, proteomic, clinical) to identify novel biological targets implicated in diseases that were previously u

In [18]:
# Turn 3 - Compaction should trigger after this turn!
await run_session(
    research_runner_compacting,
    "Tell me more about the second development you found.",
    "compaction_demo",
)


 ### Session: compaction_demo

User > Tell me more about the second development you found.
gemini-2.5-flash-lite >  You're referring to the second development I mentioned: **Advances in Protein Folding and Structure Prediction**, specifically highlighting **AlphaFold and Similar Technologies**.

Let me elaborate on why this is such a significant development in drug discovery:

**The Fundamental Problem:**

*   **Proteins are the Workhorses of Life:** Proteins are complex molecules that carry out a vast array of functions in our bodies – acting as enzymes, structural components, signaling molecules, and much more.
*   **Structure Dictates Function:** A protein's specific three-dimensional shape (its structure) is absolutely critical for its function. If a protein misfolds, it can lose its ability to work correctly, leading to disease. Conversely, understanding a protein's structure is paramount for designing drugs that can interact with it.
*   **The "Protein Folding Problem" was a Gra

In [19]:
# Turn 4
await run_session(
    research_runner_compacting,
    "Who are the main companies involved in that?",
    "compaction_demo",
)


 ### Session: compaction_demo

User > Who are the main companies involved in that?
gemini-2.5-flash-lite >  You're asking about the companies involved in **AI-driven protein folding and structure prediction**, particularly in the context of drug discovery. This is a fascinating space with a few key players and a growing ecosystem.

Here are the main categories and some prominent examples:

**1. The Innovator (and Public Data Provider):**

*   **DeepMind (an Alphabet company):**
    *   **Role:** The pioneer that brought us **AlphaFold**. Their breakthrough was the initial development of the highly accurate AI model.
    *   **Contribution:** Their decision to make the AlphaFold Protein Structure Database publicly available has been transformative. This means millions of researchers, even those without direct access to sophisticated AI, can benefit from these predicted structures.
    *   **Impact:** They provided the foundational technology and the crucial open data resource that has 

#### Verify Compaction in the Session History

In [20]:
# Get the final session state
final_session = await session_service.get_session(
    app_name=research_runner_compacting.app_name,
    user_id=USER_ID,
    session_id="compaction_demo",
)

print("--- Searching for Compaction Summary Event ---")
found_summary = False
for event in final_session.events:
    # Compaction events have a 'compaction' attribute
    if event.actions and event.actions.compaction:
        print("\n SUCCESS! Found the Compaction Event:")
        print(f"  Author: {event.author}")
        pprint(f"\n Compacted information: {event}")
        found_summary = True
        break

if not found_summary:
    print(
        "\n No compaction event found. Try increasing the number of turns in the demo."
    )

--- Searching for Compaction Summary Event ---

 SUCCESS! Found the Compaction Event:
  Author: user
('\n'
 ' Compacted information: model_version=None content=None '
 'grounding_metadata=None partial=None turn_complete=None finish_reason=None '
 'error_code=None error_message=None interrupted=None custom_metadata=None '
 'usage_metadata=None live_session_resumption_update=None '
 'input_transcription=None output_transcription=None avg_logprobs=None '
 'logprobs_result=None cache_metadata=None citation_metadata=None '
 "invocation_id='606f50db-b510-44c1-80d4-8639728c19ff' author='user' "
 'actions=EventActions(skip_summarization=None, state_delta={}, '
 'artifact_delta={}, transfer_to_agent=None, escalate=None, '
 'requested_auth_configs={}, requested_tool_confirmations={}, '
 "compaction={'start_timestamp': 1762985020.488872, 'end_timestamp': "
 "1762985027.159691, 'compacted_content': {'parts': [{'function_call': None, "
 "'code_execution_result': None, 'executable_code': None, 'file

## Working with Session State

### Creating custom tools for Session state management

In [21]:
# identify a transferable characteristic, like a user's name and their country
# create tools to capture and save it

In [22]:
# Define scope levels for state keys (following best practices)
USER_NAME_SCOPE_LEVELS = ("temp", "user", "app")

# This demonstrates how tools can write to session state using tool_context.
# The 'user:' prefix indicates this is user-specific data.
def save_userinfo(
    tool_context: ToolContext, user_name: str, country: str) -> Dict[str, Any]:
    """
    Tool to record and save user name and country in session state.

    Args:
        user_name: The username to store in session state
        country: The name of the user's country
    """
    # Write to session state using the 'user:' prefix for user data
    tool_context.state["user:name"] = user_name
    tool_context.state["user:country"] = country

    return {"status": "success"}


# This demonstrates how tools can read from session state.
def retrieve_userinfo(tool_context: ToolContext) -> Dict[str, Any]:
    """
    Tool to retrieve user name and country from session state.
    """
    # Read from session state
    user_name = tool_context.state.get("user:name", "Username not found")
    country = tool_context.state.get("user:country", "Country not found")

    return {"status": "success", "user_name": user_name, "country": country}

In [23]:
# Configuration
APP_NAME = "agents"
USER_ID = "default"
MODEL_NAME = "gemini-2.5-flash-lite"

# Create an agent with session state tools
root_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="""A text chatbot.
    Tools for managing user context:
    * To record username and country when provided use `save_userinfo` tool. 
    * To fetch username and country when required use `retrieve_userinfo` tool.
    """,
    tools=[save_userinfo, retrieve_userinfo],  # Provide the tools to the agent
)

# Set up session service and runner
session_service = InMemorySessionService()
runner = Runner(agent=root_agent, 
                session_service=session_service, 
                app_name=APP_NAME)

In [24]:
# Test conversation demonstrating session state
await run_session(
    runner,
    [
        "Hi there, how are you doing today? What is my name?",  # Agent shouldn't know the name yet
        "My name is Alex. I'm from Czech Republic.",  # Provide name - agent should save it
        "What is my name? Which country am I from?",  # Agent should recall from session state
    ],
    "state-demo-session",
)


 ### Session: state-demo-session

User > Hi there, how are you doing today? What is my name?
gemini-2.5-flash-lite >  Hello! I'm doing well, thank you for asking. I can't seem to recall your name. Could you please tell me what it is?

User > My name is Alex. I'm from Czech Republic.
gemini-2.5-flash-lite >  Hello Alex from Czech Republic! It's nice to meet you.

User > What is my name? Which country am I from?
gemini-2.5-flash-lite >  Your name is Alex and you are from Czech Republic.


In [25]:
# Retrieve the session and inspect its state
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="state-demo-session"
)

print("Session State Contents:")
print(session.state)

Session State Contents:
{'user:name': 'Alex', 'user:country': 'Czech Republic'}


### Session State Isolation

In [26]:
# Start a completely new session - the agent won't know our name
await run_session(
    runner,
    ["Hi there, how are you doing today? What is my name?"],
    "new-isolated-session",
)


 ### Session: new-isolated-session

User > Hi there, how are you doing today? What is my name?
gemini-2.5-flash-lite >  Hello! I'm doing great. I can't recall your name just yet. Can you remind me? 



### Cross-Session State Sharing

In [27]:
# Check the state of the new session
session = await session_service.get_session(
    app_name=APP_NAME, 
    user_id=USER_ID, 
    session_id="new-isolated-session"
)

print("New Session State:")
print(session.state)

New Session State:
{'user:name': 'Alex', 'user:country': 'Czech Republic'}
